# Reproducing Komori et al. (2026) from DANDI:001425: IC count, window, VIF and SHAP

**Paper.** Komori Y, Mizusaki K, Tomita K, Yoneda T, Matoba K, Morita T (2026). Mesoscopic cortical
activities associated with pupil-linked perceptions inferred via explainable machine learning.
bioRxiv, [doi:10.64898/2026.05.27.726399](https://doi.org/10.64898/2026.05.27.726399). The claims
file (`claims/komori-2026.yaml`) was written from the preprint posted June 1, 2026, with its
Methods, legends and Supplementary Figures 3 to 6.

**Dandiset.** [DANDI:001425](https://dandiarchive.org/dandiset/001425/0.250705.0947), version
0.250705.0947 ("BraiDyn-BC: mouse cued-lever-pull task dataset", Kondo et al. 2025). The paper
cites the dandiset without a version. Two versions are published, 0.250514.0602 (1814 assets) and
0.250705.0947 (1838 assets); the most recent is pinned, as the protocol asks when no version is
cited. The paper was posted in 2026, after both, so either could be the one it read. The data check
shows that 6 of the 9 files read here are the same assets in both versions; the other 3 were
replaced in the later version, and the earlier ones were not compared.
The paper uses only the resting-state recordings of this dandiset; its own recordings
(somatosensory and visual stimulation) are not deposited.

**What is tested.** The six resting-state claims. The paper's pipeline is rebuilt from its
Methods: hemodynamic correction, SVD and JADE ICA of the cortical images, pupil labels around the
dilation peak of each 40 s epoch, a GRU decoder with five-fold cross-validation and null models
trained on labels shuffled within epochs. The open details R1 to R10 of the claims file's opening
comment are followed as it names them; where several readings are named, the notebook runs the
main one and as many others as time allows, and says which were not run.

This notebook tests the IC count and window, the VIF and the SHAP claims, and runs the costly
variant of rest-decoding-above-null (R4 b, every frame of the epoch a sample), three restarts of
the ICA and the check of the SVD scaling. The decoding claims
are in `reproduce_komori_2026_decoding.ipynb`; the loading and decomposition here are the same.

| Claim | Notebook | Status |
|---|---|---|
| rest-decoding-above-null | decoding | tested (R4 b and ICA restarts in the features notebook) |
| rest-loss-accuracy-vs-null | decoding | tested |
| rest-needs-0-1-hz | decoding | tested |
| rest-ic-count-and-window | features | tested |
| rest-vif-below-10 | features | tested |
| rest-shap-medial-anterior-first | features | tested |
| fig1-*, fig2-*, fig3-*, fig4-shap-*, fig4-mapped-shap-regions, supp2-*, supp4-*, supp8-*, supp10-* (18 claims) | none | blocked: the stimulation recordings are not deposited |

In [ ]:
import pupilwf as pw                       # first: it pins the numerical kernels (pw.KERNELS)

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_info

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("kernels:", pw.KERNELS, "; linear algebra:", sorted({(i['internal_api'], i.get('architecture')) for i in threadpool_info()}, key=str))

started = time.time()
assets = pw.list_assets()
rows = pw.resting_table(assets)
chosen = pw.choose_sessions(rows)          # R1 (a): three mice, first three resting sessions each
chosen_rows = [r for mouse in chosen for r in mouse]
sessions = pw.load_sessions(chosen_rows)
print(f"streamed and reduced {len(sessions)} sessions from DANDI:{pw.DANDISET} version {pw.VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
t0 = time.time()
D = pw.decompose(sessions, hemo="fit_hp_subtract_raw")       # R2 (a), main reading
comp = pw.compactness(D["maps"], D["mask"])
# features for VIF-based selection: every frame of the 13 epochs, 0.1 Hz low-pass (R6)
def vif_frames(ics, cutoff=0.1):
    out = []
    for s, ic in zip(sessions, ics):
        a, b = pw.epochs_of(s["t"])[0][0], pw.epochs_of(s["t"])[-1][1]
        out.append(pw.normalise_features(ic, cutoff)[a:b])
    return np.vstack(out)
F01 = vif_frames(D["ics"])
keep10, removed10 = pw.select_ics(F01, comp, 10, "kept")                  # VIF of the paper's code (uncentred)
keep10c, removed10c = pw.select_ics(F01, comp, 10, "kept", centred=True)  # variant: VIF with an intercept
print(f"decomposition in {time.time() - t0:.0f} s; JADE converged in {D['sweeps']} sweeps; kept ICs ({len(keep10)}) {[int(k) for k in keep10]}; "
      f"with the centred VIF ({len(keep10c)}) {[int(k) for k in keep10c]}")

## Data check

The full data check is in the decoding notebook; this repeats what this notebook uses.

In [ ]:
print("Common cortical grid points (atlas grid, step", pw.GRID_STEP, "atlas pixels):", int(D["mask"].sum()))
print("SVD components available:", D["n_svd"], "(paper: 400; the corrected data of each recording have rank 34)")
print("Cumulative variance at 10, 30, 200 components:", np.round(D["cum_var"][[9, 29, min(199, D['n_svd'] - 1)]], 4),
      "(paper: > 99.5% at 200)")
print("Frames used to estimate the decomposition (every fifth 40 s epoch):", D["n_sel_frames"])
print("IC compactness (share of squared weight in the largest supra-half-maximum region):")
print(pd.Series(comp).round(3).to_string())
print("VIF of all 30 ICs (0.1 Hz features): max", round(float(pw.vif(F01).max()), 2),
      "uncentred (no intercept, the paper's code);", round(float(pw.vif(F01, centred=True).max()), 2), "centred")
print("Kept for 10 ICs (reading a), uncentred VIF:", [int(k) for k in keep10], "removed for VIF:", [int(k) for k in removed10],
      "max VIF of the kept:", round(float(pw.vif(F01[:, keep10]).max()), 2))
print("Kept for 10 ICs (reading a), centred VIF:", [int(k) for k in keep10c], "removed for VIF:", [int(k) for k in removed10c],
      "max VIF of the kept:", round(float(pw.vif(F01[:, keep10c], centred=True).max()), 2))
roi = sessions[0]["roi_atlas"]
names = sessions[0]["roi_names"]
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, j in zip(axes.flat, keep10):
    m = D["maps"][j]
    lim = np.nanmax(np.abs(m))
    ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.contour(roi.any(axis=0), [0.5], colors="k", linewidths=0.5)
    ax.set_title(f"IC {j}  comp {comp[j]:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("Spatial maps of the 10 kept ICs on the common atlas grid (colour limits: +/- max |weight|)")
plt.tight_layout()
plt.show()

In [ ]:
ds_main = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1)
lab = pd.DataFrame(ds_main["info"])
print("Epochs per recording:", lab.groupby("rec").size().unique(), "; samples per epoch window:", sorted(lab.n.unique()))
print("Positive frames per epoch window (of", lab.n.max(), "), by recording:")
print(lab.groupby(["subject", "day"]).n_pos.describe()[["mean", "min", "50%", "max"]].round(1).to_string())
print("Epochs whose window has no positive frame:", int((lab.n_pos == 0).sum()), "of", len(lab),
      "; all frames positive:", int((lab.n_pos == lab.n).sum()))
print("Samples:", ds_main["X"].shape, "positive share", round(float(ds_main["y"].mean()), 3))
print("Feature range:", float(ds_main["X"].min()), float(ds_main["X"].max()))
lab5 = pd.DataFrame(pw.make_dataset(sessions, D["ics"], keep10, 0.1, selection="rule")["info"])
print("Epochs kept by the selection rule R5 (b):", int(lab5.keep.sum()), "of", len(lab5))
print("Epochs whose pupil peak is at the epoch's first frame or within 0.5 s of its end (the label window then",
      "reaches into the next epoch):", int(lab.edge.sum()), "of", len(lab), "; among the epochs kept by R5 (b):",
      int((lab.edge & lab5.keep).sum()))

## rest-ic-count-and-window

> For resting data, decoding with 10 ICs and a 1.5 s feature window gave the highest median AUC
> among the IC counts (7, 10, 13) and window lengths (0.5 to 10 s) tested, though no comparison was
> significant.

IC counts under reading (a), the number kept, and (b), the number before VIF pruning (VIF of the
paper's code); windows of 0.5, 1.0, 1.5, 2.0, 3.0, 5.0 and 10.0 s with the ICs of the 10-IC count. 1.5 s is 46 frames,
the paper's number for resting data (Methods, Machine learning, Overview); the other windows are
rounded from seconds at 30 Hz, as the paper gives no frame counts for them. Main reading
combination otherwise, under both epoch selections (R5 a and b), at this claim's seed.

In [ ]:
SEED4 = pw.seed_of("rest-ic-count-and-window")
ic_sets = {}
for reading, rd in (("a", "kept"), ("b", "initial")):
    for n in (7, 10, 13):
        ic_sets[(reading, n)] = pw.select_ics(F01, comp, n, rd)
ic_rows, ic_cv, by_set = [], {}, {}
for sel in ("none", "rule"):
    for (reading, n), (keep, removed) in ic_sets.items():
        key = (sel, tuple(sorted(int(k) for k in keep)))
        if key not in by_set:      # the same IC set is fitted once
            by_set[key] = pw.run_cv(pw.make_dataset(sessions, D["ics"], keep, cutoff=0.1, selection=sel), SEED4)
        cv = by_set[key]
        ic_cv[(sel, reading, n)] = cv
        ic_rows.append(dict(R5="a" if sel == "none" else "b", reading=reading, n_ics=n, kept=len(keep), removed_for_vif=len(removed),
                            ics=sorted(int(k) for k in keep),
                            median_auc=float(cv.model_val_auc.median()), mean_auc=float(cv.model_val_auc.mean()),
                            null_mean_auc=float(cv.null_val_auc.mean()), p_vs_null=pw.paired(cv, "val_auc")["p"],
                            min_auc=float(cv.model_val_auc.min()), max_auc=float(cv.model_val_auc.max())))
it = pd.DataFrame(ic_rows)
print(it.round(4).to_string(index=False))
for sel in ("none", "rule"):
    for reading in ("a", "b"):
        ref = ic_cv[(sel, reading, 10)].model_val_auc
        ps = [stats.ranksums(ref, ic_cv[(sel, reading, n)].model_val_auc).pvalue for n in (7, 13)]
        sub = it[(it.reading == reading) & (it.R5 == ("a" if sel == "none" else "b"))]
        best = sub.loc[sub.median_auc == sub.median_auc.max(), "n_ics"].tolist()
        print(f"R5 {'a' if sel == 'none' else 'b'}, reading {reading}: highest median at {best} ICs; "
              f"Wilcoxon rank-sum 10 against 7 and 13, Holm:", np.round(multipletests(ps, method='holm')[1], 4))

In [ ]:
win_rows, win_cv = [], {}
for sel in ("none", "rule"):
    for w_s in (0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0):
        w = int(round(w_s * pw.FS)) + (1 if w_s == 1.5 else 0)   # 1.5 s is the paper's 46 frames
        t0 = time.time()
        ds = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, window=w, selection=sel)
        cv = ic_cv[(sel, "a", 10)] if w == pw.WINDOW else pw.run_cv(ds, SEED4)
        win_cv[(sel, w_s)] = cv
        win_rows.append(dict(R5="a" if sel == "none" else "b", window_s=w_s, frames=w, samples=len(ds["y"]),
                             median_auc=float(cv.model_val_auc.median()), mean_auc=float(cv.model_val_auc.mean()),
                             null_mean_auc=float(cv.null_val_auc.mean()), p_vs_null=pw.paired(cv, "val_auc")["p"],
                             s=round(time.time() - t0)))
wt = pd.DataFrame(win_rows)
print(wt.round(4).to_string(index=False))
for sel in ("none", "rule"):
    sub = wt[wt.R5 == ("a" if sel == "none" else "b")]
    ref = win_cv[(sel, 1.5)].model_val_auc
    others = [w for w in (0.5, 1.0, 2.0, 3.0, 5.0, 10.0)]
    ps = [stats.ranksums(ref, win_cv[(sel, w)].model_val_auc).pvalue for w in others]
    print(f"R5 {'a' if sel == 'none' else 'b'}: highest median AUC at window", sub.loc[sub.median_auc == sub.median_auc.max(), "window_s"].tolist(), "s;",
          "Wilcoxon rank-sum 1.5 s against", others, "Holm:", np.round(multipletests(ps, method="holm")[1], 4))

**Result: not reproduced (resting on assumptions).** With the VIF of the paper's code few ICs pass
the pruning: reading (a) keeps 7, 7 and 9 ICs for the counts 7, 10 and 13 (1, 3 and 5 removed), and
reading (b) keeps 6, 7 and 8. Under reading (a) the 7- and 10-IC sets are the same 7 ICs, and
reading (b)'s 10-IC set is that set too. Under R5 a, 13 ICs give the highest median AUC under both
readings (reading a 0.669, against 0.614 for 7 and 10; reading b 0.676, against 0.627 for 7 and
0.614 for 10). Under R5 b, reading (a) has 7 and 10 ICs tied highest (0.539, the same model; 13 ICs
0.471) and reading (b) 7 ICs highest (0.588; 10 ICs 0.539, 13 ICs 0.440). No rank-sum test of 10 ICs
against 7 or 13 is significant (Holm p >= 0.69), as in the paper. For the window, the highest median
is at 10 s under R5 a (0.6403, against 0.6137 at 1.5 s) and at 0.5 s under R5 b (0.5476, against
0.5394); the medians differ by less than 0.03 between windows, and no Holm-corrected test is
significant (all p = 1.0). 10 ICs are highest only in a tie with 7 ICs that are the same set, which
says nothing about the count, and 1.5 s is never highest: under the criterion this is not
reproduced. The outcome verified in round 2, partially reproduced, rested on the round-2 run, whose
decomposition was set by rounding (see the restarts below); the verifier agreed with the change in
round 3. Two of these models differ from their null models at p < 0.05, both with the model lower:
13 ICs under reading (b), under R5 a (0.659 against 0.698, p = 0.021) and under R5 b (0.489 against
0.608, p = 0.015).

## rest-vif-below-10

> With 10 ICs the resting-state features have a maximum variance inflation factor of 9.0 across
> features and time frames, below the VIF < 10 bound; 7 ICs gave a significantly lower maximum VIF,
> 13 and 15 ICs did not differ from 10. (Expected 9.0 +/- 0.9.)

VIF within each time frame of the validation samples, maximum over features and frames, per model
(fold); the folds are those of rest-decoding-above-null. The VIF is computed as the paper's code
computes it (`#07_vif_calc.ipynb`, statsmodels `variance_inflation_factor` on the features with no
constant column, so without an intercept); the VIF with an intercept (centred) is shown as a
variant, and so is the code's choice of training samples in place of the Methods' validation
samples. IC sets as in rest-ic-count-and-window, both readings, plus 15 ICs, under R5 a and b. With the
VIF of the paper's code the 10-IC count keeps 7 ICs under both readings, so the 10-IC values are
those of these 7.
Deterministic given the folds.

In [ ]:
SEED1 = pw.seed_of("rest-decoding-above-null")
def max_vif_per_fold(keep, selection="none", centred=False, part="val"):
    ds = pw.make_dataset(sessions, D["ics"], keep, cutoff=0.1, selection=selection)
    units = ds["unit"]
    by_mouse = {}
    for u, m in zip(units, ds["mouse"]):
        by_mouse.setdefault(m, set()).add(int(u))
    fold = pw.folds_by_mouse({m: sorted(v) for m, v in sorted(by_mouse.items())}, SEED1)
    fs = np.array([fold[int(u)] for u in units])
    out = []
    for f in range(5):
        Xv = ds["X"][fs == f] if part == "val" else ds["X"][fs != f]
        out.append(max(float(pw.vif(Xv[:, k, :], centred).max()) for k in range(Xv.shape[1])))
    return np.array(out)
vif_rows, vif_vals = [], {}
for sel, centred, part in (("none", False, "val"), ("rule", False, "val"), ("none", True, "val"), ("rule", True, "val"),
                           ("none", False, "train")):
    for reading, rd in (("a", "kept"), ("b", "initial")):
        for n in (7, 10, 13, 15):
            keep, removed = pw.select_ics(F01, comp, n, rd)
            v = max_vif_per_fold(keep, sel, centred, part)
            vif_vals[(sel, centred, part, reading, n)] = v
            vif_rows.append(dict(R5="a" if sel == "none" else "b", vif="centred" if centred else "paper", samples=part,
                                 reading=reading, n_ics=n, kept=len(keep), max_over_models=v.max(), median=np.median(v),
                                 per_model=np.round(v, 2).tolist()))
vtab = pd.DataFrame(vif_rows)
print(vtab.round(3).to_string(index=False))
for sel, centred, part in (("none", False, "val"), ("rule", False, "val"), ("none", True, "val"), ("rule", True, "val"),
                           ("none", False, "train")):
    for reading in ("a", "b"):
        ref = vif_vals[(sel, centred, part, reading, 10)]
        ps, dirs = [], []
        for n in (7, 13, 15):
            ps.append(stats.mannwhitneyu(vif_vals[(sel, centred, part, reading, n)], ref, alternative="two-sided").pvalue)
            dirs.append(float(np.median(vif_vals[(sel, centred, part, reading, n)]) - np.median(ref)))
        print(f"R5 {'a' if sel == 'none' else 'b'}, {'centred' if centred else 'paper'} VIF, {part} samples, reading {reading}: "
              f"10-IC max {ref.max():.2f}; Mann-Whitney U against 10 ICs (7, 13, 15), Holm:",
              np.round(multipletests(ps, method="holm")[1], 4), "; median difference", np.round(dirs, 3))

**Result: not reproduced (resting on assumptions).** With the VIF the paper's code computes (no
intercept), on the validation samples under R5 a, the maximum for the 10-IC count (7 ICs kept) is
21.55 over the five models (per model 9.99 to 21.55): above 10 and outside 9.0 +/- 0.9. With an
intercept it is 2.97, and on the training samples, which the paper's code uses, 10.32 (per model
8.85 to 10.32); neither is within 9.0 +/- 0.9. Under R5 b the validation folds hold 212 to 530
samples from 17 epochs, and the maximum reaches 1.3e6 with either VIF. For the comparisons of the
claim, on the validation samples (R5 a, the paper's VIF) the 7-IC count is not lower (reading a: the
same set; reading b Holm p = 0.67), and 13 and 15 ICs do not differ from 10 (Holm p >= 0.67). On the
training samples, reading (b), the 7-IC count (6 kept) has a lower median (8.74 against 10.07), not
significantly (Holm p = 0.095); under reading (a), 15 ICs (10 kept) are higher (Holm p = 0.024),
where the paper finds no difference. On the frames the selection uses, the 7 kept ICs have a
maximum VIF of 9.99 (cell output above), below 10 by construction.

## rest-shap-medial-anterior-first

> In the resting-state model, SHAP values of an IC in the medial anterior cortex (IC7) rise before
> those of the medial posterior region (IC6) and the secondary motor cortex (IC4).

The models of rest-decoding-above-null are fitted again here (same data, folds and seeds), under R5 a and R5 b.
Attribution: integrated gradients from the mean validation input (see the helper; Deep SHAP's
DeepExplainer does not support PyTorch GRU layers), for the first validation sample of each
validation epoch whose prediction is positive [assumption: the Methods' "first sample per experiment" is read with the 40 s epoch as the experiment, the unit the null shuffle and the folds also use; per recording would explain 9 samples in all]: at 0.5 (Methods reading) or at the model's Youden
threshold (code reading). SHAP is averaged over these samples per model, and the 5 models are
compared with the 5 null models per IC and 0.2 s window by paired t-test, uncorrected. The three ICs
are matched by their maps: medial anterior (largest share of positive weight in a band of 0.5 mm
on each side of the midline anterior to RSP, excluding RSP), RSP (RSPd and RSPagl) and MOs.

In [ ]:
from sklearn.metrics import roc_curve
ds_by = {"R5 a": ds_main, "R5 b": pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, selection="rule")}
models_by = {}
for combo, ds in ds_by.items():
    cv_s, models_by[combo] = pw.run_cv(ds, SEED1, keep_models=True)
    r = pw.paired(cv_s, "val_auc")
    print(combo, "refitted models, validation AUC:", np.round(cv_s.model_val_auc, 3).tolist(),
          "null:", np.round(cv_s.null_val_auc, 3).tolist(), f"; mean {r['model']:.3f} against {r['null']:.3f}, p = {r['p']:.3g}")
def attributions(reading, ds, models_s):
    per_model = {"model": [], "null": []}
    counts = []
    for net, nnet, va in models_s:
        Xv, yv, uv = ds["X"][va], ds["y"][va], ds["unit"][va]
        base = Xv.mean(axis=0)
        for kind, m in (("model", net), ("null", nnet)):
            p = pw.predict(m, Xv)
            if reading == "methods":
                thr = 0.5
            else:
                fpr, tpr, th = roc_curve(yv, p)
                thr = th[np.argmax(tpr - fpr)]
            idx = [np.flatnonzero((uv == u) & (p >= thr))[0] for u in np.unique(uv) if ((uv == u) & (p >= thr)).any()]
            counts.append(dict(kind=kind, n=len(idx), thr=float(thr)))
            if not idx:
                per_model[kind].append(np.full(Xv.shape[1:], np.nan))
                continue
            sh = pw.integrated_gradients(m, Xv[idx], base)
            per_model[kind].append(sh.mean(axis=0))   # frames x ICs
    return np.array(per_model["model"]), np.array(per_model["null"]), pd.DataFrame(counts)
shap = {(combo, r): attributions(r, ds_by[combo], models_by[combo]) for combo in ds_by for r in ("methods", "code")}
for (combo, r), (sm, sn, cnt) in shap.items():
    print(combo, r, "samples explained per model:", cnt[cnt.kind == "model"].n.tolist(), "null:", cnt[cnt.kind == "null"].n.tolist())

In [ ]:
# Match the three ICs by their maps.
roi = sessions[0]["roi_atlas"]
names = sessions[0]["roi_names"]
def region(prefixes):
    return np.any([roi[i] for i, n in enumerate(names) if any(n.startswith(p) for p in prefixes)], axis=0)
rsp, mos = region(["RSP"]), region(["MOs"])
left = region([n for n in names if n.endswith("_l")])
right = region([n for n in names if n.endswith("_r")])
cols = np.arange(roi.shape[2])
xl = (left * cols).sum() / left.sum()
xr = (right * cols).sum() / right.sum()
rows_idx = np.arange(roi.shape[1])
rsp_rows = rows_idx[rsp.any(axis=1)]
mid = (xl + xr) / 2
# grid pixel size: atlas pixels per data pixel (affine scale) times the data grid spacing
um_per_grid = pw.GRID_STEP * np.sqrt(abs(np.linalg.det(sessions[0]["affine"][:, :2]))) * 18.18
band = np.abs(cols - mid)[None, :] * um_per_grid <= 500
ant_dir = -1 if (region(["MOs"]).any(axis=1) * rows_idx).sum() / mos.sum() < rsp_rows.mean() else 1
anterior = (rows_idx[:, None] < rsp_rows.min()) if ant_dir == -1 else (rows_idx[:, None] > rsp_rows.max())
cortex = roi.any(axis=0)
medant = band & anterior & cortex & ~rsp
def share(j, area):
    m = np.nan_to_num(D["maps"][j])
    pk = m.flat[np.argmax(np.abs(m))]
    m = np.clip(m * np.sign(pk), 0, None) ** 2
    return float(m[area].sum() / m.sum())
sh_tab = pd.DataFrame([{"ic": j, "medial_anterior": share(j, medant), "RSP": share(j, rsp), "MOs": share(j, mos)} for j in keep10])
print(f"midline at grid column {mid:.1f}; {um_per_grid:.1f} um per grid point; medial anterior band {int(medant.sum())} grid points")
print(sh_tab.round(3).to_string(index=False))
ic7 = int(sh_tab.loc[sh_tab.medial_anterior.idxmax(), "ic"])
rest_tab = sh_tab[sh_tab.ic != ic7]
ic6 = int(rest_tab.loc[rest_tab.RSP.idxmax(), "ic"])
ic4 = int(rest_tab[rest_tab.ic != ic6].loc[lambda d: d.MOs.idxmax(), "ic"])
match = {"medial anterior (IC7)": ic7, "RSP (IC6)": ic6, "MOs (IC4)": ic4}
print("matched:", match)
# Whether the IC matched as medial anterior lies mostly in the medial anterior band (the criterion's
# "no IC matches" case when it does not): its share there against its share in RSP and in MOs.
r7 = sh_tab.set_index("ic").loc[ic7]
print(f"IC {ic7}: medial anterior share {r7.medial_anterior:.3f}, RSP {r7.RSP:.3f}, MOs {r7.MOs:.3f}; "
      f"largest in the medial anterior band: {bool(r7.medial_anterior > max(r7.RSP, r7.MOs))}")
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, (k, j) in zip(axes, match.items()):
    m = D["maps"][j]
    lim = np.nanmax(np.abs(m))
    ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.contour(cortex, [0.5], colors="k", linewidths=0.5)
    ax.contour(medant, [0.5], colors="g", linewidths=0.8)
    ax.set_title(f"{k}: IC {j}", fontsize=9)
    ax.axis("off")
plt.suptitle("Matched ICs (green: medial anterior band); colour limits +/- max |weight|")
plt.tight_layout()
plt.show()

In [ ]:
edges = np.round(np.arange(-1.5, 0.0001, 0.2), 1)
t_frames = (np.arange(pw.WINDOW) - (pw.WINDOW - 1)) / pw.FS          # -1.5 ... 0 s
wins = [(-1.5, -1.3), (-1.3, -1.1), (-1.1, -0.9), (-0.9, -0.7), (-0.7, -0.5), (-0.5, -0.3), (-0.3, -0.1), (-0.1, 0.0)]
pos = {j: keep10.index(j) for j in match.values()}
out = []
for (combo, r), (sm, sn, cnt) in shap.items():
    for k, j in match.items():
        c = pos[j]
        onset = None
        rowp = {}
        for (a, b) in wins:
            sel = (t_frames >= a - 1e-9) & ((t_frames < b - 1e-9) if b < 0 else (t_frames <= b + 1e-9))
            x, y = np.nanmean(sm[:, sel, c], axis=1), np.nanmean(sn[:, sel, c], axis=1)
            ok = np.isfinite(x) & np.isfinite(y)          # a model with no positive prediction has no SHAP
            t, p = stats.ttest_rel(x[ok], y[ok]) if ok.sum() >= 3 else (np.nan, np.nan)
            rowp[f"{a}..{b}"] = round(float(p), 3) if np.isfinite(p) else None
            if onset is None and np.isfinite(p) and p < 0.05 and x.mean() > y.mean():
                onset = a
        mean_curve = np.nanmean(sm[:, :, c], axis=0)
        out.append(dict(R5=combo[-1], reading=r, ic=k, pairs=int(ok.sum()), onset_a=onset, peak_b=float(t_frames[np.argmax(mean_curve)]),
                        peak_value=float(mean_curve.max()), peak_at_edge=bool(np.argmax(mean_curve) in (0, pw.WINDOW - 1)),
                        **rowp))
st = pd.DataFrame(out)
print("p-values per window (model against null, paired t over 5 models), onset (a) = first window with p < 0.05 "
      "and model above null; peak (b) = time of the maximum of the mean SHAP over models")
print(st.to_string(index=False))
for combo, r in shap:
    s = st[(st.reading == r) & (st.R5 == combo[-1])].set_index("ic")
    for col in ("onset_a", "peak_b"):
        v = s[col]
        a = v["medial anterior (IC7)"]
        prec = [o for o in ("RSP (IC6)", "MOs (IC4)") if a is not None and pd.notna(a) and pd.notna(v[o]) and a < v[o]]
        print(f"{combo}, {r}, {col}: medial anterior {a}, RSP {v['RSP (IC6)']}, MOs {v['MOs (IC4)']}; precedes: {prec}")

**Result: not reproduced (resting on assumptions).** No kept IC matches the medial anterior
description, the criterion's "no IC matches" case. The IC with the largest share of its positive
weight in the medial anterior band, IC 18, has 0.266 there against 0.494 in MOs, so its map lies
mostly in MOs (`largest in the medial anterior band: False`); no kept IC has more than 0.27 in the
band. The RSP and MOs matches are IC 28 (0.160 in RSP, the largest among the kept) and IC 4 (0.574
in MOs). The timing is reported all the same, and it does not favour the medial anterior IC either.
Under R5 a no window has the model's SHAP significantly above the null's for any of the three ICs,
so reading (a) gives no onset; the RSP IC differs from null at -0.3 to -0.1 s and -0.1 to 0.0 s (p =
0.018 to 0.043), and the MOs IC under the code reading at -0.5 to -0.3 s (p = 0.035), each time with
the model's SHAP the lower. Under reading (b) the mean SHAP of each of the three peaks at the last
frame (0.0 s), except the RSP IC under the code reading (-0.033 s), so the medial anterior IC
precedes neither. Under R5 b the Methods reading explains a sample in one model only, and the code
reading 1 to 3 per model; no window differs from null, and the peaks are at 0.0 s for the medial
anterior and MOs ICs and at -0.17 and -0.20 s for RSP. Every medial anterior peak is at the window's
last frame (`peak_at_edge`).

## rest-decoding-above-null, reading R4 b, and ICA restarts

The costly variant of the decoding claim: every frame of every epoch is a sample, positive only
inside the window around the peak (the claims file's R4 reading b), under R5 b with the deposited
diameter (R3 a) and the keypoint width (R3 b), each under the decoding claim's seed and four more;
where one is above null at the claim's seed, it is also run at the six cutoffs of
rest-needs-0-1-hz. Under R5 a it takes 140,400 samples and about 10 minutes per run, and is not run
here for lack of time.

Then the decomposition itself: JADE started from three other points (the whitened data rotated by
a random orthogonal matrix; seeds are the claim's seed plus 200, 201 and 202, a standard choice),
with the ICs selected by the same rule, and the main reading (R5 a) refitted on each. Last, the
scaling of the SVD temporal components: the ratio of the 30th to the 31st eigenvalue of the
covariance that jadeR reduces to 30 dimensions, with the components as the paper's code forms them
(main) and divided by their singular values as the Methods write.

In [ ]:
t0 = time.time()
r4_rows, r4_cv = [], {}
for name, kw in (("R4 b, R5 b", dict(selection="rule")),
                 ("R4 b, R5 b, R3 b", dict(selection="rule", pupil_key="pupil_kp"))):
    ds_b = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, sample_reading="epoch", **kw)
    cv_b = pw.run_cv(ds_b, SEED1)
    r4_cv[name] = cv_b
    print(name, "samples", len(ds_b["y"]), "positive share", round(float(ds_b["y"].mean()), 4))
    print(cv_b[["fold", "model_val_auc", "null_val_auc", "model_val_loss", "null_val_loss", "model_val_acc", "null_val_acc"]].round(3).to_string(index=False))
    row = dict(reading=name, samples=len(ds_b["y"]))
    for m in ("val_auc", "val_loss", "val_acc"):
        r = pw.paired(cv_b, m)
        row.update({m + "_model": r["model"], m + "_null": r["null"], m + "_p": r["p"]})
        print(f"{m}: model {r['model']:.3f}, null {r['null']:.3f}, t = {r['t']:.2f}, p = {r['p']:.3g}")
    r4_rows.append(row)
# Both R4 b combinations under the claim's seed and four more; where one is above null at the claim's
# seed, also at the six cutoffs (seed of rest-needs-0-1-hz), which that claim asks for.
SEED3 = pw.seed_of("rest-needs-0-1-hz")
s_rows, c_rows = [], []
for name, kw in (("R4 b, R5 b", dict(selection="rule")),
                 ("R4 b, R5 b, R3 b", dict(selection="rule", pupil_key="pupil_kp"))):
    ds_b = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, sample_reading="epoch", **kw)
    for off in (0, 1000, 2000, 3000, 4000):
        cv = r4_cv[name] if off == 0 else pw.run_cv(ds_b, SEED1 + off)
        r = pw.paired(cv, "val_auc")
        s_rows.append(dict(reading=name, seed=f"claim+{off}", model_auc=r["model"], null_auc=r["null"], t=r["t"], p=r["p"],
                           above=bool(r["p"] < 0.05 and r["model"] > r["null"]), null_below_half=int((cv.null_val_auc < 0.5).sum())))
    p0 = pw.paired(r4_cv[name], "val_auc")
    for c in ((0.05, 0.1, 0.2, 0.5, 1.0, 2.0) if p0["p"] < 0.05 and p0["model"] > p0["null"] else ()):
        cv = pw.run_cv(pw.make_dataset(sessions, D["ics"], keep10, cutoff=c, sample_reading="epoch", **kw), SEED3)
        r = pw.paired(cv, "val_auc")
        c_rows.append(dict(reading=name, cutoff_hz=c, model_auc=r["model"], null_auc=r["null"], p=r["p"],
                           above_null=bool(r["p"] < 0.05 and r["model"] > r["null"]), aucs=np.round(cv.model_val_auc, 3).tolist()))
print("\nR4 b under five seeds (the claim's seed plus 0, 1000, 2000, 3000, 4000):")
print(pd.DataFrame(s_rows).round(4).to_string(index=False))
r4c = pd.DataFrame(c_rows, columns=["reading", "cutoff_hz", "model_auc", "null_auc", "p", "above_null", "aucs"])
print("\nR4 b at the six cutoffs (seed of rest-needs-0-1-hz):" if len(r4c) else "\nNo R4 b combination is above null at the claim's seed; the cutoffs are not run.")
if len(r4c):
    print(r4c.round(4).to_string(index=False))
for name in r4c.reading.unique():
    g = r4c[r4c.reading == name].set_index("cutoff_hz")
    others = [0.05, 0.2, 0.5, 1.0, 2.0]
    ps = [stats.ranksums(g.loc[0.1, "aucs"], g.loc[c, "aucs"]).pvalue for c in others]
    print(name, ": Wilcoxon rank-sum, 0.1 Hz against", dict(zip(others, np.round(multipletests(ps, method="holm")[1], 4))), "(Holm)")
print(f"({time.time() - t0:.0f} s; notebook {time.time() - started:.0f} s)")

In [ ]:
t0 = time.time()
rs_rows = []
for i in range(3):
    Dr = pw.decompose(sessions, hemo="fit_hp_subtract_raw", rotation_seed=SEED1 + 200 + i)
    comp_r = pw.compactness(Dr["maps"], Dr["mask"])
    Fr = vif_frames(Dr["ics"])
    keep_r, removed_r = pw.select_ics(Fr, comp_r, 10, "kept")
    # how many of the kept ICs have a close match (|r| of the maps > 0.9) among the main decomposition's kept ICs
    mm = np.nan_to_num(D["maps"][keep10].reshape(len(keep10), -1))
    mr = np.nan_to_num(Dr["maps"][keep_r].reshape(len(keep_r), -1))
    cc = np.abs(np.corrcoef(mr, mm)[:len(keep_r), len(keep_r):])
    for sel in ("none",):
        cv = pw.run_cv(pw.make_dataset(sessions, Dr["ics"], keep_r, cutoff=0.1, selection=sel), SEED1)
        r = pw.paired(cv, "val_auc")
        a, b = pw.paired(cv, "val_loss"), pw.paired(cv, "val_acc")
        rs_rows.append(dict(restart=i, R5="a" if sel == "none" else "b", kept=len(keep_r), removed=len(removed_r),
                            matched_to_main=int((cc.max(axis=1) > 0.9).sum()),
                            model_auc=r["model"], null_auc=r["null"], p=r["p"],
                            loss_model=a["model"], loss_null=a["null"], loss_p=a["p"],
                            acc_model=b["model"], acc_null=b["null"], acc_p=b["p"]))
print(pd.DataFrame(rs_rows).round(4).to_string(index=False))
# The Methods' scaling (each SVD temporal component divided by its singular value) for comparison: on
# the frames the SVD is fitted to, the components then have equal variance, and the 30 of 200
# dimensions jadeR keeps are not set by the data (ratio of the 30th to the 31st eigenvalue near 1).
D_m = pw.decompose(sessions, hemo="fit_hp_subtract_raw", scaling="methods")
print(f"30th / 31st eigenvalue of the covariance jadeR reduces: {D['eig_gap']:.4f} with the code's scaling (main), "
      f"{D_m['eig_gap']:.8f} with the Methods' scaling")
print(f"({time.time() - t0:.0f} s; notebook {time.time() - started:.0f} s)")

**Result of the R4 b runs and the restarts, for rest-decoding-above-null.** With every frame of the
17 selected epochs a sample (R4 b with R5 b; 20,400 samples, 3.6% positive) the models reach 0.660
against 0.634 for the null models (p = 0.817), and with the keypoint pupil (R3 b; 27,600 samples)
0.603 against 0.598 (p = 0.925). Neither is above null at any of the five seeds (p = 0.058 to 0.82,
and 0.075 to 0.92), so the cutoffs are not run under R4 b. The validation loss is not lower than the
null's at p < 0.05 (p = 0.131 and 0.171), and model and null have the same accuracy in every fold.
These runs add nothing to the two combinations above null in the decoding notebook, which decide the
outcome, partially reproduced and seed sensitive.

Two of the three restarts of JADE land on the same 7 ICs as the main decomposition and give the same
models; the third keeps 9 ICs, of which 5 match the main ones (|r| of the maps > 0.9), and its
models do not differ significantly from their null models (0.583 against 0.646, p = 0.144). With the
components scaled as in the paper's code, the 30 dimensions jadeR keeps are set by the data
(eigenvalue ratio 1.0188); divided by their singular values, as the Methods write, the ratio is
1.00000000, and which 30 of the 200 dimensions are kept is then set by rounding. That is why the
round-2 run of this entry, which used the Methods' scaling, kept other ICs than the run before it on
another machine.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| rest-decoding-above-null | partially reproduced (assumed; seed sensitive; above null in two of 16 combinations under R5 b, one of them at 4 of 5 seeds; mean AUC at most 0.72) |
| rest-loss-accuracy-vs-null | not reproduced (assumed) |
| rest-needs-0-1-hz | not reproduced (assumed; the claim's pattern holds in 1 of the 4 combinations run at the cutoffs) |
| rest-ic-count-and-window | not reproduced (assumed; 10 ICs highest only in a tie with the same 7-IC set, 1.5 s never) |
| rest-vif-below-10 | not reproduced (assumed) |
| rest-shap-medial-anterior-first | not reproduced (assumed; no kept IC matches the medial anterior description) |
| 18 claims on the paper's stimulation recordings | blocked: data missing |

**How the analysis differs from the paper's.** The paper does not say which three mice or which
resting sessions it used; the first three mice recorded, with their first three resting sessions,
are used, and all 25 mice (R1 b) were not run (about 90 sessions to stream, some three hours). The
ICs it kept "by visual inspection" are replaced by a repeatable rule (the most compact maps), then
pruned with the VIF as the paper's code computes it (no intercept); with that VIF only 7 ICs pass
where the paper keeps 10. Frames are binned 2 x 2 (144 x 144) and the decomposition is made on a
common atlas grid; the SVD keeps 306 components, the rank of the corrected data, in place of 400.
The SVD temporal components keep their singular values, as in the paper's code: divided by them,
as the Methods write, they have equal variance, and the 30 dimensions jadeR keeps out of 200 are
then chosen by rounding, so that two machines gave two different sets of ICs. JADE runs to
convergence (an earlier cap of 100 sweeps stopped it short), and the numerical kernels are pinned. The decomposition is then the same on every machine tried,
but the GRU fits are not always: the final run, on an AMD Zen 5 processor, gave some fits that differ
from those of the round-2 run, and the numbers here are the final run's. The GRU is written in PyTorch with Keras'
defaults, and SHAP is computed by integrated gradients from the mean input in place of
DeepExplainer. Failed pupil detections are replaced. The epoch selection is run under both
readings (none, and the stimulus data's rule); the other readings are run one at a time under the
first and in every combination of R2, R3, R7 and R8 under the second, with R4 b under the second
only.

**What was found about the dataset.** The files hold everything the resting analysis needs: frames
of both channels, pupil diameter and keypoints on the imaging clock, atlas ROIs and the atlas
transform. The excitation wavelengths of the two imaging planes are swapped in the metadata. The
deposited pupil diameter holds unflagged failed detections (values up to 2.9e21 pixels) in two of
the nine sessions read. Six of the 98 resting files hold no behaviour and so no pupil.